In [ ]:
import re

import json

INPUT_FILE = "./result/word/predictions_gemma-2-2b_finetuned_test_word.txt"
OUTPUT_FILE = "./result/word/predictions_gemma-2-2b_finetuned_test_word_cleaned.txt"


def clean_sql(sql):
    if not sql:
        return sql

    sql = str(sql).strip()

    # Keep from first SELECT
    m = re.search(r"\bselect\b", sql, re.IGNORECASE)
    if m:
        sql = sql[m.start():]

    # Stop when Gemma starts generating prompt/schema again
    stop_patterns = [
        r"\bmodel\b",
        r"\[schema\]",
        r"create\s+table",
        r"\busermodel\b",
        r"\benderror\b",
        r"\[bài tập\]",
    ]

    end = len(sql)

    for pattern in stop_patterns:
        m = re.search(pattern, sql, re.IGNORECASE)
        if m:
            end = min(end, m.start())

    sql = sql[:end]

    # Normalize spaces
    sql = re.sub(r"\s+", " ", sql)

    # Fix operators
    sql = re.sub(r">\s+=", ">=", sql)
    sql = re.sub(r"<\s+=", "<=", sql)
    sql = re.sub(r"!\s+=", "!=", sql)

    # Remove spaces before punctuation
    sql = re.sub(r"\s+,", ",", sql)
    sql = re.sub(r"\(\s+", "(", sql)
    sql = re.sub(r"\s+\)", ")", sql)

    return sql.strip()


with open(INPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

for item in data:
    # Chỉ clean predict để giữ nguyên ground truth
    if "predict_sql" in item:
        item["predict_sql"] = clean_sql(item["predict_sql"])

    # Nếu muốn clean cả gold thì bỏ comment
    # if "gold_sql" in item:
    #     item["gold_sql"] = clean_sql(item["gold_sql"])


with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print(f"Saved {len(data)} examples to {OUTPUT_FILE}")

In [ ]:
!git clone https://github.com/taoyds/spider.git

In [7]:
import sys
from pathlib import Path            

ROOT = Path.cwd()      # Thư mục VITEXT2SQL
SPIDER = ROOT / "spider"

sys.path.insert(0, str(SPIDER))

print(ROOT)
print(SPIDER)

c:\Users\ADMIN\Desktop\ViText2SQL
c:\Users\ADMIN\Desktop\ViText2SQL\spider


In [9]:
import glob
import json
import os

import spider.process_sql as ps
from spider.evaluation import Evaluator, build_foreign_key_map_from_json

TABLE_PATH = "data/syllable-level/tables.json"


ps.load_tables_json(TABLE_PATH)

kmaps = build_foreign_key_map_from_json(TABLE_PATH)
evaluator = Evaluator()


def evaluate(pred_file):

    with open(pred_file, encoding="utf-8") as f:
        data = json.load(f)

    correct = 0

    for sample in data:

        db_id = sample["db_id"]

        schema = ps.Schema(ps.get_schema(db_id))

        try:

            g_sql = ps.get_sql(schema, sample["gold_sql"])
            p_sql = ps.get_sql(schema, sample["predict_sql"])

            if evaluator.eval_exact_match(p_sql, g_sql):
                correct += 1

        except Exception as e:
            print(db_id, e)

    print(
        os.path.basename(pred_file),
        correct,
        "/",
        len(data),
        "=",
        correct / len(data)
    )



evaluate("result/syllable\predictions_gpt-5.4_few_shot_test.txt")

assets_maintenance 'nhật'
book_2 'ấn'
company_1 'người'
course_teach 'giáo'
course_teach 'giáo'
course_teach 'giáo'
decoration_competition 'thành'
match_season 'quốc'
match_season 'cầu'
match_season 'trận'
match_season 'quốc'
perpetrator 'cá'
perpetrator 'cá'
perpetrator 'cá'
station_weather 'tàu'
station_weather 'tàu'
wedding 'cá'
activity_1 'giảng'
activity_1 'giảng'
activity_1 'hoạt'
activity_1 'tham'
body_builder 'người'
city_record 'trận'
city_record 'thành'
city_record 'thành'
college_1 'nhân'
college_1 Error col: địa
college_1 'khoá'
college_1 'lớp'
college_1 'lớp'
college_1 'sinh'
college_1 'lớp'
college_1 'lớp'
college_1 'giáo'
college_1 'sinh'
college_1 'lớp'
college_1 'khoá'
cre_Doc_Control_Systems 'đại'
cre_Drama_Workshop_Groups 'địa'
cre_Drama_Workshop_Groups 'địa'
cre_Drama_Workshop_Groups 'hoá'
customers_and_invoices 'khách'
customers_and_invoices 'tài'
customers_and_invoices 'giao'
customers_and_invoices 'giao'
customers_and_invoices 'sản'
customers_and_invoices 'đơn'
c